# Feature Engineering - NO2

En este notebook construimos las features nuevas a partir del dataset final (`train.parquet`), 
pensadas para sacarle mas partido al modelo de regresion que prediga episodios de alta contaminacion.

**Que vamos a hacer:**
1. Cargar los datos desde `../data_sample/`
2. Split cronologico train_fe / test_fe (sin tocar el train/test del notebook 02)
3. Features temporales (ciclicas, estacion del anio, finde)
4. Lags y medias moviles de NO2 por estacion
5. Imputacion de meteo con flag de missing
6. Target encoding de estacion
7. Pipeline completo: fit en train_fe, transform en test_fe

Trabajamos siempre evitando fuga de datos (data leakage): todo lo que "aprende" de los datos 
(medianas, medias por estacion) se calcula solo con `train_fe`, nunca con `test_fe`.

## 1. Imports

Librerias necesarias para cargar los datos y montar el pipeline de feature engineering.

In [1]:
# Imports generales + para el pipeline de feature engineering
import glob
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin  # clases base para crear transformers custom compatibles con sklearn
from sklearn.pipeline import Pipeline  # para encadenar todos los pasos de FE en uno solo

## 2. Carga de datos

Leemos los ficheros parquet desde `../data_sample/`. Usamos `glob` con un patron comodin 
por si hay mas de un fichero parquet en esa carpeta.

In [2]:
# Recogemos el dataset completo.
df = pd.read_parquet('../data_sample/dataset_final.parquet') 

print(df.shape)
df.head()

(51001, 16)


,estacion,fecha,no2,temperatura,humedad,viento_vel,precipitacion,intensidad_mean,intensidad_max,ocupacion_mean,carga_mean,vmed_mean,vmed_max,tipo_elem_moda,distancia_meteo_km,distancia_trafico_km
0,4,2021-01-01,10,4.070833,NaN,NaN,NaN,388.155556,916.0,3.522222,21.722222,0.0,0.0,URB,0.0,0.06
1,4,2021-01-02,23,2.387500,NaN,NaN,NaN,516.344828,924.0,4.574713,28.563218,0.0,0.0,URB,0.0,0.06
2,4,2021-01-03,29,2.233333,NaN,NaN,NaN,476.076923,924.0,4.109890,26.329670,0.0,0.0,URB,0.0,0.06
3,4,2021-01-04,29,3.016667,NaN,NaN,NaN,677.088235,998.0,5.911765,37.044118,0.0,0.0,URB,0.0,0.06
4,4,2021-01-05,54,0.250000,NaN,NaN,NaN,588.095745,922.0,4.978723,32.202128,0.0,0.0,URB,0.0,0.06


## 3. Split cronologico: train_fe / test_fe

Este split es nuevo y va en paralelo al `train`/`test` aleatorio del notebook 02 (no lo sustituye).

**Por que un split por fecha y no aleatorio:** vamos a calcular lags y medias moviles de NO2 
(el valor de ayer, la media de los ultimos 7 dias...). Si el split fuera aleatorio, filas de 
fechas muy cercanas podrian caer una en train y otra en test, y un lag podria colarse informacion 
del futuro. Cortando por fecha, todo lo que hay en `test_fe` es estrictamente posterior a `train_fe`.

Usamos el percentil 80 de las fechas como corte, así queda aproximadamente un split 80/20.

In [3]:
# Fecha de corte: percentil 80 de las fechas disponibles (deja ~80/20)
fechas_unicas = sorted(df['fecha'].unique())
fecha_corte = fechas_unicas[int(len(fechas_unicas) * 0.8)]

train_fe = df[df['fecha'] < fecha_corte].copy()
test_fe = df[df['fecha'] >= fecha_corte].copy()

print(f"Fecha de corte: {fecha_corte}")
print(f"train_fe: {train_fe.shape[0]} filas ({train_fe['fecha'].min()} a {train_fe['fecha'].max()})")
print(f"test_fe:  {test_fe.shape[0]} filas ({test_fe['fecha'].min()} a {test_fe['fecha'].max()})")
print(f"Proporcion test_fe: {test_fe.shape[0] / df.shape[0]:.2%}")

Fecha de corte: 2023-09-25 00:00:00
train_fe: 40729 filas (2019-01-01 00:00:00 a 2023-09-24 00:00:00)
test_fe:  10272 filas (2023-09-25 00:00:00 a 2024-11-30 00:00:00)
Proporcion test_fe: 20.14%


## 4. Features temporales

Ampliamos el mes/dia de semana/finde que ya se vio en el notebook 03, con codificacion ciclica 
(seno/coseno) para que diciembre y enero queden "cerca" numericamente, y anadimos la estacion del anio.

Este transformer no aprende nada de los datos (no calcula ningun estadistico), solo deriva 
columnas nuevas a partir de la fecha. Por eso `fit` no hace nada.

In [4]:
class FeaturesTemporales(BaseEstimator, TransformerMixin):
    # No aprende nada de los datos (no hay estadisticos que calcular),
    # asi que fit no hace nada y solo devuelve self, como exige la interfaz de sklearn
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()  # copia defensiva, para no modificar el DataFrame original por referencia

        # Codificacion ciclica del mes: en vez de 1-12, usamos seno/coseno
        # para que diciembre (12) y enero (1) queden "cerca" en el espacio numerico,
        # en vez de parecer los extremos opuestos de una escala lineal
        X['mes_sin'] = np.sin(2 * np.pi * X['fecha'].dt.month / 12)
        X['mes_cos'] = np.cos(2 * np.pi * X['fecha'].dt.month / 12)

        # Mismo truco pero con el dia del anio (1-365), da mas resolucion que el mes
        X['dia_anio_sin'] = np.sin(2 * np.pi * X['fecha'].dt.dayofyear / 365)
        X['dia_anio_cos'] = np.cos(2 * np.pi * X['fecha'].dt.dayofyear / 365)

        # Flag de fin de semana: dayofweek 5 y 6 son sabado y domingo
        X['finde'] = X['fecha'].dt.dayofweek.isin([5, 6]).astype(int)

        # Estacion del anio a partir del mes, usando np.select para evaluar
        # varias condiciones a la vez (mas legible que anidar np.where)
        mes = X['fecha'].dt.month
        X['estacion_anio'] = np.select(
            [mes.isin([12, 1, 2]), mes.isin([3, 4, 5]), mes.isin([6, 7, 8])],  # condiciones en orden
            ['invierno', 'primavera', 'verano'],  # valor si se cumple cada condicion
            default='otonio'  # lo que no entra en ninguna de las anteriores (9, 10, 11)
        )
        return X

## 5. Lags y medias moviles de NO2

El NO2 tiene memoria: el valor de hoy se parece mucho al de ayer. Calculamos lags 
(1 dia y 7 dias atras) y medias moviles (3 y 7 dias), siempre agrupando por `estacion` 
para no mezclar el pasado de una estacion con el de otra.

Tampoco aprende nada de los datos: `shift`/`rolling` solo miran hacia atras dentro de cada fila, 
asi que es seguro calcularlo igual en train y en test.

In [5]:
class FeaturesLagNO2(BaseEstimator, TransformerMixin):
    def __init__(self, lags=(1, 7), ventanas_rolling=(3, 7)):
        self.lags = lags  # que lags calcular: 1 dia atras, 7 dias atras (mismo dia semana anterior)
        self.ventanas_rolling = ventanas_rolling  # tamanios de ventana para las medias moviles

    # Tampoco aprende nada: shift/rolling solo miran hacia atras dentro de cada fila,
    # asi que es seguro calcularlo igual en train y en test
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ordenamos por estacion y fecha: imprescindible para que shift/rolling
        # avancen en el orden correcto dentro de cada estacion
        X = X.sort_values(['estacion', 'fecha']).copy()

        # Agrupamos por estacion para que cada lag/rolling se calcule
        # solo con los dias de esa misma estacion, sin mezclar con otras
        grupo = X.groupby('estacion')['no2']

        # Lags: el valor de no2 hace N dias, para esa misma estacion
        for lag in self.lags:
            X[f'no2_lag{lag}'] = grupo.shift(lag)

        # Rolling: media movil de los ultimos N dias, sin contar el dia actual
        # (por eso el shift(1) antes del rolling: evita que el dia de hoy
        # se use a si mismo para predecirse)
        for ventana in self.ventanas_rolling:
            X[f'no2_roll_mean_{ventana}'] = (
                X.groupby('estacion')['no2']
                .transform(
                    lambda serie: serie.shift(1).rolling(ventana).mean()
                )
            )

        return X

## 6. Imputacion de meteo con flag de missing

`viento_vel` y `precipitacion` tienen bastante missing (71% y 54%). En vez de eliminar filas, 
imputamos con la mediana y anadimos una columna flag que indica si el valor original era missing, 
por si esa ausencia es informativa en si misma.

Aqui si aprendemos algo de los datos (la mediana), y lo hacemos solo con `train_fe` en el `fit`.

In [6]:
class ImputadorMeteo(BaseEstimator, TransformerMixin):
    def __init__(self, columnas=('viento_vel', 'precipitacion')):
        self.columnas = columnas  # columnas meteo a imputar

    def fit(self, X, y=None):
        # Aqui si aprendemos algo de los datos: la mediana de cada columna,
        # pero calculada SOLO con X (que en la practica sera train_fe)
        self.medianas_ = {col: X[col].median() for col in self.columnas}
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            # Flag que guarda si el valor original era missing, ANTES de imputar
            # (si no la creamos antes de fillna, perderiamos esta informacion)
            X[f'{col}_missing'] = X[col].isna().astype(int)

            # Rellenamos los NaN con la mediana aprendida en fit
            # (en test_fe usamos la mediana de train_fe, nunca la de test_fe)
            X[col] = X[col].fillna(self.medianas_[col])
        return X

## 7. Target encoding de estacion

Cada estacion tiene un nivel de contaminacion medio distinto (zonas de mucho trafico vs 
residenciales). Codificamos cada estacion por su NO2 medio historico, calculado solo con `train_fe`.

In [7]:
class TargetEncoderEstacion(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Media global de no2, por si aparece una estacion en test
        # que no estaba en train (no deberia pasar aqui, pero es una red de seguridad)
        self.media_global_ = X['no2'].mean()

        # Media de no2 por estacion, calculada solo con train_fe
        self.medias_estacion_ = X.groupby('estacion')['no2'].mean()
        return self

    def transform(self, X):
        X = X.copy()
        # Mapeamos cada estacion a su media aprendida en fit;
        # fillna cubre el caso de una estacion nueva no vista en train
        X['estacion_no2_mean'] = (
            X['estacion'].map(self.medias_estacion_).fillna(self.media_global_)
        )
        return X

## 8. Pipeline completo

Encadenamos los 4 transformers y aplicamos `fit_transform` solo en `train_fe` (aprende y transforma 
a la vez), y `transform` en `test_fe` (aplica lo ya aprendido, sin recalcular nada con datos de test).

Asi nos aseguramos de que no hay fuga de datos entre train_fe y test_fe en ningun paso.

In [8]:
# Encadenamos los 4 transformers en el orden en que queremos que se apliquen
pipeline_fe = Pipeline([
    ('temporales', FeaturesTemporales()),        # no aprende nada
    ('lags_no2', FeaturesLagNO2()),               # no aprende nada
    ('imputador_meteo', ImputadorMeteo()),        # aprende medianas de train
    ('target_encoder', TargetEncoderEstacion()),  # aprende medias de train
])

# fit_transform en train_fe: ajusta (fit) los transformers que aprenden algo
# usando solo train_fe, y a la vez transforma train_fe con ellos
train_fe_procesado = pipeline_fe.fit_transform(train_fe)

# transform en test_fe: aplica lo YA aprendido en train_fe (medianas, medias por estacion)
# sin volver a calcular nada con datos de test_fe -> asi evitamos el leakage
test_fe_procesado = pipeline_fe.transform(test_fe)

print(train_fe_procesado.shape, test_fe_procesado.shape)
train_fe_procesado.head()

(40729, 29) (10272, 29)


,estacion,fecha,no2,temperatura,humedad,viento_vel,precipitacion,intensidad_mean,intensidad_max,ocupacion_mean,...,dia_anio_cos,finde,estacion_anio,no2_lag1,no2_lag7,no2_roll_mean_3,no2_roll_mean_7,viento_vel_missing,precipitacion_missing,estacion_no2_mean
17277,4,2019-01-01,71,4.500000,NaN,11.022917,0.0,360.708333,653.0,1.270833,...,0.999852,0,invierno,NaN,NaN,16.666667,19.714286,1,1,30.419847
17278,4,2019-01-02,84,3.970833,NaN,11.022917,0.0,505.635417,890.0,2.843750,...,0.999407,0,invierno,71.0,NaN,11.666667,18.142857,1,1,30.419847
17279,4,2019-01-03,77,4.345833,NaN,11.022917,0.0,539.322917,887.0,3.479167,...,0.998667,0,invierno,84.0,NaN,14.000000,17.571429,1,1,30.419847
17280,4,2019-01-04,86,4.050000,NaN,11.022917,0.0,557.541667,923.0,3.229167,...,0.997630,0,invierno,77.0,NaN,17.333333,17.857143,1,1,30.419847
17281,4,2019-01-05,75,3.587500,NaN,11.022917,0.0,454.812500,794.0,2.770833,...,0.996298,1,invierno,86.0,NaN,23.333333,19.142857,1,1,30.419847


In [9]:
# Guardamos train_fe_procesado y test_fe_procesado en data_sample
# (las versiones YA pasadas por el pipeline, con las features nuevas incluidas)
train_fe_procesado.to_parquet('../data_sample/train_fe.parquet', index=False)
test_fe_procesado.to_parquet('../data_sample/test_fe.parquet', index=False)

print('train_fe guardado:', train_fe_procesado.shape)
print('test_fe guardado:', test_fe_procesado.shape)

train_fe guardado: (40729, 29)
test_fe guardado: (10272, 29)


## Conclusiones

- El split `train_fe`/`test_fe` es cronologico (corte en el percentil 80 de las fechas), 
  necesario para poder calcular lags y rolling de NO2 sin fuga de datos.
- Todas las estadisticas que "aprenden" de los datos (medianas de meteo, medias de NO2 por 
  estacion) se ajustan solo con `train_fe` y se aplican igual a `test_fe` con `transform`.
- Pendiente a revisar: los primeros dias de cada estacion en `train_fe` (y tambien el primer 
  dia de `test_fe`) se quedan con lags en NaN, porque no hay "dia anterior" disponible. 
  Hay que decidir si se imputan o se eliminan antes de entrenar el modelo.